In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
#from sklearn.compose import ColumnTransfomer
#from sklearn.pipeline import pipeline
#from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_regression,f_classif
#from sklearn.decomposition import PCA
from sklearn.preprocessing import OneHotEncoder

df =pd.read_csv("/content/01 covid_toy.csv")
display(df.head())

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [2]:
# To automatically find numeric columns:
numeric_columns=df.select_dtypes(include=np.number).columns

print("Numeric Columns:")
print(numeric_columns)

#origin is often represented by numbers such as 1,2,3,
#so techinally pandas may identify it as numeric
#even though it represents categories.


Numeric Columns:
Index(['age', 'fever'], dtype='object')


In [3]:
# To automatically find categorical columns:
categorical_columns=df.select_dtypes(include="object").columns

print("Categorical Columns:")
print(categorical_columns)


Categorical Columns:
Index(['gender', 'cough', 'city', 'has_covid'], dtype='object')


In [4]:
#first check missing values:
print(df.isnull().sum())

# Auto-MPG commonly represent missing horespower value using ?.
# So first convert ? into  missing value

df=df.replace("?",np.nan)
print(df.isnull().sum())


age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64
age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64


In [6]:
scaler=MinMaxScaler().set_output(transform="pandas")

scaled_data=scaler.fit_transform(df[numeric_columns])

scaled_data.head()


,age,fever
0,0.696203,0.833333
1,0.278481,0.333333
2,0.468354,0.500000
3,0.329114,0.000000
4,0.759494,0.500000


In [8]:
categorical_columns=["gender","cough","city"]

#OHE

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
).set_output(transform="pandas")

encoded_data=encoder.fit_transform(df[categorical_columns])
display(encoded_data.head())


,gender_Female,gender_Male,cough_Mild,cough_Strong,city_Bangalore,city_Delhi,city_Kolkata,city_Mumbai
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
2,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0
3,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0


In [9]:
#suppose has_covid is our traget
x=df.drop("has_covid",axis=1)
y=df["has_covid"]


In [10]:
display(x)

,age,gender,fever,cough,city
0,60,Male,103.0,Mild,Kolkata
1,27,Male,100.0,Mild,Delhi
2,42,Male,101.0,Mild,Delhi
3,31,Female,98.0,Mild,Kolkata
4,65,Female,101.0,Mild,Mumbai
...,...,...,...,...,...
95,12,Female,104.0,Mild,Bangalore
96,51,Female,101.0,Strong,Kolkata
97,20,Female,101.0,Mild,Bangalore
98,5,Female,98.0,Strong,Mumbai


In [11]:
display(y)


,has_covid
0,No
1,Yes
2,No
3,No
4,No
...,...
95,No
96,Yes
97,No
98,No


In [12]:
cleaned_data_x=pd.concat([scaled_data,encoded_data],axis=1)


In [13]:
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)

In [14]:
y

array([0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0,
       0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1])

In [15]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test=train_test_split(cleaned_data_x,y,test_size=0.2,random_state=1)


In [16]:
from sklearn.ensemble import RandomForestClassifier

model=RandomForestClassifier()


In [17]:
model.fit(x_train,y_train)

RandomForestClassifier()

In [18]:
y_pred=model.predict(x_test)

In [19]:
y_pred

array([0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1])

In [22]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)


0.25